# Mini Project: Document Scanner

In [1]:
import cv2
import numpy as np


In [27]:
img = cv2.imread("C:/Users/Shyam/OneDrive/Pictures/sks/test.jpg")
orig = img.copy()
img = cv2.resize(img, (700, 800)) # optional resize for faster processing
cv2.imshow("Original", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [19]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5,5), 0)


In [33]:
edges = cv2.Canny(blur, 10, 20)
cv2.imshow("Edges", edges)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [34]:
contours, hierarchy = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
contours = sorted(contours, key=cv2.contourArea, reverse=True)  # largest first

for cnt in contours:
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
    if len(approx) == 4:  # document has 4 corners
        doc_cnt = approx
        break


In [36]:
cv2.drawContours(img, [doc_cnt], -1, (0,255,0), 2)
cv2.imshow("Document Contour", img)
cv2.waitKey(0)
cv2.destroyAllWindows()



In [37]:
def reorder(points):
    points = points.reshape(4, 2)
    new_points = np.zeros((4, 2), dtype=np.float32)
    s = points.sum(axis=1)
    diff = np.diff(points, axis=1)

    new_points[0] = points[np.argmin(s)]   # top-left
    new_points[2] = points[np.argmax(s)]   # bottom-right
    new_points[1] = points[np.argmin(diff)] # top-right
    new_points[3] = points[np.argmax(diff)] # bottom-left
    return new_points

doc_cnt = reorder(doc_cnt)
pts1 = np.float32(doc_cnt)
pts2 = np.float32([[0,0],[600,0],[600,800],[0,800]])
matrix = cv2.getPerspectiveTransform(pts1, pts2)
scanned = cv2.warpPerspective(orig, matrix, (600,800))

cv2.imshow("Scanned Document", scanned)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [38]:
scanned_gray = cv2.cvtColor(scanned, cv2.COLOR_BGR2GRAY)
_, scanned_thresh = cv2.threshold(scanned_gray, 150, 255, cv2.THRESH_BINARY)

cv2.imshow("Paper-Like Scan", scanned_thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [40]:
import cv2
import numpy as np

# ------------------------------
# 1. Read Image
# ------------------------------
img = cv2.imread("C:/Users/Shyam/OneDrive/Pictures/sks/test.jpg")
orig = img.copy()

# Resize for faster processing
img = cv2.resize(img, (700, 700))

# ------------------------------
# 2. Preprocessing (Gray, Blur, Edge)
# ------------------------------
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blur, 75, 200)

cv2.imshow("Edges", edges)

# ------------------------------
# 3. Find Largest Contour (Document)
# ------------------------------
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours = sorted(contours, key=cv2.contourArea, reverse=True)

document_contour = None

for cnt in contours:
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)

    # Document must have 4 points
    if len(approx) == 4:
        document_contour = approx
        break

if document_contour is None:
    print("❌ Document not found!")
    exit()

cv2.drawContours(img, [document_contour], -1, (0, 255, 0), 3)
cv2.imshow("Document Outline", img)

# ------------------------------
# 4. Reorder points
# ------------------------------
def reorder(points):
    points = points.reshape((4, 2))
    reordered = np.zeros((4, 2), dtype=np.float32)

    # top-left = smallest sum
    # bottom-right = largest sum
    sum_points = points.sum(axis=1)
    reordered[0] = points[np.argmin(sum_points)]
    reordered[3] = points[np.argmax(sum_points)]

    # top-right = min diff
    # bottom-left = max diff
    diff_points = np.diff(points, axis=1)
    reordered[1] = points[np.argmin(diff_points)]
    reordered[2] = points[np.argmax(diff_points)]

    return reordered

pts = reorder(document_contour)

# ------------------------------
# 5. Compute Output Dimensions
# ------------------------------
(tl, tr, br, bl) = pts

widthA = np.linalg.norm(br - bl)
widthB = np.linalg.norm(tr - tl)
maxWidth = max(int(widthA), int(widthB))

heightA = np.linalg.norm(tr - br)
heightB = np.linalg.norm(tl - bl)
maxHeight = max(int(heightA), int(heightB))

# Destination points (top-down view)
dst = np.array([
    [0, 0],
    [maxWidth - 1, 0],
    [maxWidth - 1, maxHeight - 1],
    [0, maxHeight - 1]
], dtype=np.float32)

# ------------------------------
# 6. Apply Perspective Transform
# ------------------------------
matrix = cv2.getPerspectiveTransform(pts, dst)
scanned = cv2.warpPerspective(orig, matrix, (maxWidth, maxHeight))

# ------------------------------
# 7. Optional: Convert to Black & White (Scanner effect)
# ------------------------------
gray_scanned = cv2.cvtColor(scanned, cv2.COLOR_BGR2GRAY)
thresh_scanned = cv2.adaptiveThreshold(
    gray_scanned, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 11, 2
)

# ------------------------------
# 8. Show Results
# ------------------------------
# cv2.imshow("Original", orig)
cv2.imshow("Scanned Output", scanned)
cv2.imshow("Scanned B&W", thresh_scanned)

cv2.waitKey(0)
cv2.destroyAllWindows()
